In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

In [2]:
e_data = pd.read_csv('energy_dataset.csv')
w_data = pd.read_csv('weather_features.csv')

In [3]:
e_data['time'] = pd.to_datetime(e_data['time'], utc=True)
w_data['dt_iso'] = pd.to_datetime(w_data['dt_iso'], utc=True)

In [4]:
# Aggregate weather from city level to country level
numerical_cols = ['temp', 'temp_min', 'temp_max', 'pressure', 
                  'humidity', 'wind_speed', 'wind_deg', 
                  'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all']

categorical_cols = ['weather_id', 'weather_main', 
                    'weather_description', 'weather_icon']

# Drop city name column
w_clean = w_data.drop(columns=['city_name'])

# Aggregate numerical — mean across 5 cities
w_numerical = w_clean.groupby('dt_iso')[numerical_cols].mean()

# Aggregate categorical — mode across 5 cities
w_categorical = w_clean.groupby('dt_iso')[categorical_cols].agg(
    lambda x: x.mode()[0]
)

# Combine both
w_final = pd.concat([w_numerical, w_categorical], 
                     axis=1).reset_index()

# Rename for merging
w_final.rename(columns={'dt_iso': 'time'}, inplace=True)

print('Weather aggregated shape:', w_final.shape)

Weather aggregated shape: (35064, 16)


In [5]:
# Create fresh ML copy from raw energy data
ml_df = e_data.copy()

# Filter years 2015-2018 only
ml_df = ml_df[ml_df['time'].dt.year.isin([2015, 2016, 2017, 2018])]

print('Energy data shape after filtering:', ml_df.shape)

# Merge with weather data
ml_df = pd.merge(ml_df,
                 w_final,
                 on='time',
                 how='left')

print('Merged shape:', ml_df.shape)

Energy data shape after filtering: (35063, 29)
Merged shape: (35063, 44)


In [6]:
# Columns to drop
cols_to_drop = [
    # High missing values (>70%)
    'forecast wind offshore eday ahead',
    'generation hydro pumped storage aggregated',
    
    # Zero or near zero values
    'generation fossil coal-derived gas',
    'generation fossil oil shale',
    'generation fossil peat',
    'generation geothermal',
    'generation marine',
    'generation wind offshore',
    
    # High cardinality categorical
    'weather_description',
    'weather_icon',
    'weather_id',
    
    # Redundant columns
    'total load forecast',  # redundant with total load actual
    'temp_min',             # redundant with temp
    'temp_max',             # redundant with temp
    'forecast wind onshore day ahead',  # data leakage risk
    'forecast solar day ahead',         # data leakage risk
    'price day ahead',                  # data leakage risk
]

ml_df = ml_df.drop(columns=cols_to_drop, errors='ignore')

print('Shape after dropping columns:', ml_df.shape)
print('Remaining columns:')
print(ml_df.columns.tolist())

Shape after dropping columns: (35063, 27)
Remaining columns:
['time', 'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption', 'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation nuclear', 'generation other', 'generation other renewable', 'generation solar', 'generation waste', 'generation wind onshore', 'total load actual', 'price actual', 'temp', 'pressure', 'humidity', 'wind_speed', 'wind_deg', 'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'weather_main']


In [7]:
ml_df['total load actual'] = ml_df['total load actual'].fillna(
    e_data['total load forecast']
)

ml_df = ml_df.dropna(subset=['price actual'])

In [8]:
ml_df['hour'] = ml_df['time'].dt.hour
ml_df['day_of_week'] = ml_df['time'].dt.dayofweek
ml_df['month'] = ml_df['time'].dt.month
ml_df['is_weekend'] = ml_df['day_of_week'].isin([5, 6]).astype(int)

print('Shape after time features:', ml_df.shape)
print('New columns added:', ['hour', 'day_of_week', 'month', 'is_weekend'])

Shape after time features: (35063, 31)
New columns added: ['hour', 'day_of_week', 'month', 'is_weekend']


In [9]:
ml_df = ml_df.sort_values('time')

# Lag features
ml_df['price_lag_24'] = ml_df['price actual'].shift(24)
ml_df['price_lag_168'] = ml_df['price actual'].shift(168)

# Drop NaN rows created by lag
ml_df = ml_df.dropna()

# Drop time column — no longer needed
ml_df = ml_df.drop(columns=['time'])

print('Shape after lag features:', ml_df.shape)

Shape after lag features: (34879, 32)


In [10]:
# One hot encode weather_main
ml_df = pd.get_dummies(ml_df,
                        columns=['weather_main'],
                        prefix='weather',
                        drop_first=True)

print('Shape after encoding:', ml_df.shape)
print('Weather columns:', [col for col in ml_df.columns 
                           if col.startswith('weather')])

Shape after encoding: (34879, 38)
Weather columns: ['weather_clouds', 'weather_drizzle', 'weather_fog', 'weather_haze', 'weather_mist', 'weather_rain', 'weather_thunderstorm']


In [11]:
# Test-Train split
X = ml_df.drop(columns=['price actual'])
y = ml_df['price actual']

# Split — 80% train, 20% test
# shuffle=False — important for time series data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (27903, 37)
X_test shape: (6976, 37)
y_train shape: (27903,)
y_test shape: (6976,)


In [12]:
#Fill missing data

numerical_cols = X_train.select_dtypes(include=['number']).columns
train_median = X_train[numerical_cols].median()

X_train[numerical_cols] = X_train[numerical_cols].fillna(train_median)

X_test[numerical_cols] = X_test[numerical_cols].fillna(train_median)

# Feature scaling for Linear regression

# Separate copies for Random Forest and XGBoost
X_train_tree = X_train.copy()  
X_test_tree = X_test.copy()

# Separate copy for Linear Regression
X_train_linear = X_train.copy()  
X_test_linear = X_test.copy()

# Scale only Linear Regression data
scaler = StandardScaler()

X_train_linear = pd.DataFrame(
    scaler.fit_transform(X_train_linear),
    columns=X_train.columns
)

X_test_linear = pd.DataFrame(
    scaler.transform(X_test_linear),
    columns=X_test.columns
)

print('Scaling complete')
print('X_train_linear shape:', X_train_linear.shape)

Scaling complete
X_train_linear shape: (27903, 37)


In [13]:
import os
os.makedirs('data', exist_ok=True)

# Data for tree based datasets
X_train_tree.to_csv('data/X_train_tree.csv', index=False)
X_test_tree.to_csv('data/X_test_tree.csv', index=False)

# Data for linear regression datasets
X_train_linear.to_csv('data/X_train_linear.csv', index=False)
X_test_linear.to_csv('data/X_test_linear.csv', index=False)


y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print('All datasets saved successfully')

All datasets saved successfully
